In [1]:
import mlflow
import pandas as pd
import requests
import json
import textstat
import numpy as np
from sentence_transformers import util
from transformers import AutoTokenizer, AutoModel
import torch
import textstat
from sklearn.metrics import precision_score, recall_score, accuracy_score


c:\Users\pedro\Documents\SB-100_Squad4\Mlflow-Squad4\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"

In [3]:
mlflow.set_tracking_uri(uri="http://localhost:5000")

In [4]:
QDRANT_URL = "https://sb100qdrant.optin.com.br"
API_KEY = "fwaoNYhMTH3vf2QfzrxajQ=="
COLLECTION_NAME = "sb100"

def search_chunks(query, k=3):

    query_vector = embed(query).tolist()

    url = f"{QDRANT_URL}/collections/{COLLECTION_NAME}/points/search"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "vector": {
            "name": "vetor_denso",
            "vector": query_vector
        },
        "limit": k,
        "with_payload": True
    }

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()

    results = response.json()["result"]

    chunks = []
    chunk_infos = []

    for r in results:
        content = r["payload"].get("content", "")

        chunks.append(content)

        chunk_infos.append({
            "id": r.get("id"),
            "score": r.get("score"),
            "file": r["payload"].get("file"),
            "chunk_index": r["payload"].get("chunk_index"),
            "preview": content[:200]
        })

    return chunks, chunk_infos

In [5]:
PROMPT_TEMPLATE = """
Olá, sou o Agente SB100, seu assistente científico especializado.

Use apenas o CONTEXTO fornecido para responder à pergunta.

REGRAS:
- Não utilize conhecimento externo
- Não invente informações
- Seja direto, objetivo e técnico
- Máximo 5 linhas
- Sempre cite o(s) documento(s) utilizado(s)
- Se não encontrar a resposta no contexto, responda exatamente:
  "Não encontrei essa informação nos documentos fornecidos."

CONTEXTO:
{context}

PERGUNTA:
{pergunta}

Formato da resposta:

Resposta:
Texto técnico, objetivo e curto

Fonte:
Documento(s) utilizado(s)
"""

In [6]:
def ollama_predict(df, PROMPT_TEMPLATE, top_k=3, min_words=50):
    outputs = []
    traces = []

    url = "http://localhost:11434/api/generate"

    for _, row in df.iterrows():
        pergunta = row["inputs"]

        chunks, chunk_infos = search_chunks(pergunta, k=top_k*2)

        filtered_chunks = []
        filtered_infos = []
        for chunk, info in zip(chunks, chunk_infos):
            text_lower = chunk.lower()
            if len(chunk.split()) >= min_words and not any(x in text_lower for x in ["informações gerais", "sumário", "conteúdo", "índice", "tabela"]):
                filtered_chunks.append(chunk)
                filtered_infos.append(info)

        sorted_chunks = sorted(zip(filtered_chunks, filtered_infos), key=lambda x: x[1].get("score", 0), reverse=True)
        sorted_chunks = sorted_chunks[:top_k]

        if sorted_chunks:
            chunks, chunk_infos = zip(*sorted_chunks)
        else:
            chunks, chunk_infos = [], []

        context_parts = []
        for chunk, info in zip(chunks, chunk_infos):
            context_parts.append(
                f"Documento: {info['file'].replace('.pdf','')}\n"
                f"Score: {info.get('score', 0):.3f}\n"
                f"Texto: {chunk}\n"
            )
        context = "\n\n".join(context_parts)

        sources = "\n".join([
            f"- {info['file'].replace('.pdf','')}"
            for info in chunk_infos
        ])

        prompt_final = PROMPT_TEMPLATE.format(
            pergunta=pergunta,
            context=context,
            sources=sources
        )

        print("======== PERGUNTA ========")
        print(pergunta)
        print("======== CONTEXTO ========")
        print(context[:1000])
        print("======== SOURCES ========")
        print(sources)

        payload = {
            "model": "gemma3:4b",
            "prompt": prompt_final,
            "temperature": 0.1,
            "stream": False
        }

        r = requests.post(url, json=payload)
        r.raise_for_status()

        response_json = r.json()
        completion = response_json.get("response", "")

        for info in chunk_infos:
            chunk_str = f"Chunk: {info['chunk_index']}"
            doc_name = info['file'].replace('.pdf','')
            completion = completion.replace(chunk_str, f"Documento: {doc_name}")

        if not completion.strip().lower().startswith("olá, sou o agente sb100"):
            completion = "Olá, sou o Agente SB100. " + completion.strip()

        completion = completion.replace("Resposta:", "").strip()

        outputs.append(completion)

        traces.append({
            "question": pergunta,
            "chunks": chunk_infos,
            "context": context
        })

    return outputs, traces

In [7]:
eval_data = pd.DataFrame(
       {
    "inputs": [
        "Qual é a recomendação de calagem para pomares de citrus visando corrigir a acidez do solo?"
    ],
    "ground_truth": [
        "."
    ]
}

)


In [8]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B")
model = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-0.6B")

def embed(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)

    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings[0].float().numpy()

def simple_token_metrics(pred, gt):
    pred_tokens = set(pred.lower().split())
    gt_tokens = set(gt.lower().split())

    tp = len(pred_tokens & gt_tokens)

    precision = tp / len(pred_tokens) if pred_tokens else 0
    recall = tp / len(gt_tokens) if gt_tokens else 0
    accuracy = 1 if pred.strip().lower() == gt.strip().lower() else 0

    return precision, recall, accuracy


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 375.26it/s, Materializing param=norm.weight]                              


In [2]:
with mlflow.start_run(run_name="gemma3:4b_RAG_Citrus"):

    mlflow.log_param("model_name", "gemma3:4b")
    mlflow.log_param("rag", True)

    predictions, traces = ollama_predict(eval_data, PROMPT_TEMPLATE)

    similarities = []
    ari_scores = []
    precisions = []
    recalls = []
    accuracies = []
    chunk_scores = []

    for pred, gt, trace in zip(predictions, eval_data["ground_truth"], traces):

        emb_p = torch.tensor(embed(pred))
        emb_g = torch.tensor(embed(gt))
        similarities.append(util.cos_sim(emb_p, emb_g).item())

        ari_scores.append(textstat.automated_readability_index(pred))

        p, r, a = simple_token_metrics(pred, gt)
        precisions.append(p)
        recalls.append(r)
        accuracies.append(a)

        scores = [c["score"] for c in trace["chunks"] if c["score"] is not None]
        if scores:
            chunk_scores.append(np.mean(scores))

    mlflow.log_metric("answer_similarity", float(np.mean(similarities)))
    mlflow.log_metric("ari_grade", float(np.mean(ari_scores)))
    mlflow.log_metric("precision", float(np.mean(precisions)))
    mlflow.log_metric("recall", float(np.mean(recalls)))
    mlflow.log_metric("accuracy", float(np.mean(accuracies)))

    if chunk_scores:
        mlflow.log_metric("avg_chunk_score", float(np.mean(chunk_scores)))

    mlflow.log_dict(traces, "rag_traces.json")

    eval_table = pd.DataFrame({
        "inputs": eval_data["inputs"],
        "ground_truth": eval_data["ground_truth"],
        "prediction": predictions,
        "answer_similarity": similarities,
        "ari_grade": ari_scores,
        "precision": precisions,
        "recall": recalls,
        "accuracy": accuracies
    })

    mlflow.log_table(eval_table, "eval/evaluation_table.json")

    for i, out in enumerate(predictions):
        mlflow.log_text(out, f"outputs/output_{i}.txt")

NameError: name 'mlflow' is not defined